# Notebook 01 — Data Preparation & Validation

**Purpose:** Load Dataset.xlsx, fix the date column (Excel serial → datetime), validate all 17 features, verify lag/cyclical encoding calculations, mark ML-ready rows, and export `data/processed/tourism_dataset_final.csv`.

This CSV is the single source of truth for all subsequent notebooks.

---

## 1. Imports & Paths

In [ ]:
import pandas as pd
import numpy as np
import os

RAW_XLSX = '../data/raw/Dataset.xlsx'
OUT_CSV  = '../data/processed/tourism_dataset_final.csv'
HIST_CSV = '../backend/data/historical.csv'

os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../backend/data',   exist_ok=True)
print('Imports OK')

## 2. Load Dataset

In [ ]:
df = pd.read_excel(RAW_XLSX, sheet_name='Dataset')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

## 3. Fix Date Column

Excel stores dates as integer serial numbers (days since 1900-01-00).
We convert them to proper `datetime` objects.

In [ ]:
if pd.api.types.is_numeric_dtype(df['date']):
    df['date'] = pd.to_datetime('1899-12-30') + pd.to_timedelta(df['date'].astype(int), unit='D')
else:
    df['date'] = pd.to_datetime(df['date'])

# Normalise to month-start (1st of each month)
df['date'] = df['date'].dt.to_period('M').dt.to_timestamp()
print('Date range:', df['date'].min(), 'to', df['date'].max())
print('Total months:', len(df))

## 4. Type Fixes & Null Check

In [ ]:
# Replace string 'None' with actual NaN in shock_type
df['shock_type'] = df['shock_type'].replace('None', None)

numeric_cols = [
    'tourist_arrivals', 'usd_lkr_rate_monthly',
    'gtrends_flights', 'gtrends_hotels', 'gtrends_visa',
    'shock_flag', 'month_sin', 'month_cos',
    'arrivals_lag_1', 'arrivals_lag_3', 'arrivals_lag_12',
    'arrivals_rolling_mean_3'
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print('=== Missing Values ===')
print(df.isnull().sum())
print()
df.describe().round(2)

## 5. Distribution Checks

In [ ]:
print('shock_flag counts:')
print(df['shock_flag'].value_counts())
print()
print('shock_type breakdown:')
print(df['shock_type'].value_counts(dropna=False))
print()
print('dataset_split breakdown:')
print(df['dataset_split'].value_counts())

## 6. Verify Feature Engineering

Recompute lags and cyclical encoding from scratch and compare with stored values.
If all checks pass, the stored values are correct.

In [ ]:
df = df.sort_values('date').reset_index(drop=True)

lag1_expected  = df['tourist_arrivals'].shift(1)
lag12_expected = df['tourist_arrivals'].shift(12)
sin_expected   = np.sin(2 * np.pi * df['month'] / 12)
cos_expected   = np.cos(2 * np.pi * df['month'] / 12)

# Compare (skip first rows where lag is NaN)
valid_idx = df.index[df['arrivals_lag_1'].notna()]
lag1_ok  = np.allclose(df.loc[valid_idx, 'arrivals_lag_1'], lag1_expected[valid_idx], equal_nan=True)
sin_ok   = np.allclose(df['month_sin'], sin_expected, atol=1e-6)
cos_ok   = np.allclose(df['month_cos'], cos_expected, atol=1e-6)

print(f'Lag-1  check: {"PASS" if lag1_ok else "FAIL"}')
print(f'sin    check: {"PASS" if sin_ok  else "FAIL"}')
print(f'cos    check: {"PASS" if cos_ok  else "FAIL"}')

## 7. Mark ML-Ready Rows

The first 12 rows have NaN in `arrivals_lag_12`.
We flag which rows have ALL features available for the ML notebooks.

In [ ]:
ml_feature_cols = [
    'usd_lkr_rate_monthly', 'gtrends_flights', 'gtrends_hotels', 'gtrends_visa',
    'shock_flag', 'month_sin', 'month_cos',
    'arrivals_lag_1', 'arrivals_lag_3', 'arrivals_lag_12', 'arrivals_rolling_mean_3'
]
df['ml_ready'] = df[ml_feature_cols].notna().all(axis=1).astype(int)
print(f'ML-ready rows: {df["ml_ready"].sum()} of {len(df)}')

## 8. Save Processed Dataset

In [ ]:
final_cols = [
    'date', 'year', 'month', 'tourist_arrivals',
    'usd_lkr_rate_monthly',
    'gtrends_flights', 'gtrends_hotels', 'gtrends_visa',
    'shock_flag', 'shock_type',
    'month_sin', 'month_cos',
    'arrivals_lag_1', 'arrivals_lag_3', 'arrivals_lag_12',
    'arrivals_rolling_mean_3',
    'dataset_split', 'ml_ready'
]
df_out = df[final_cols].copy()

df_out.to_csv(OUT_CSV,  index=False)
df_out.to_csv(HIST_CSV, index=False)   # copy served by FastAPI later

print(f'Saved to : {OUT_CSV}')
print(f'API copy : {HIST_CSV}')
print(f'Shape    : {df_out.shape}')
df_out.tail()

## 9. Summary

In [ ]:
print('=== FINAL DATASET SUMMARY ===')
print(f'Date range   : {df_out["date"].min().date()} to {df_out["date"].max().date()}')
print(f'Total months : {len(df_out)}')
print(f'Train months : {(df_out["dataset_split"]=="train").sum()}')
print(f'Test months  : {(df_out["dataset_split"]=="test").sum()}')
print(f'Shock months : {int(df_out["shock_flag"].sum())}')
print(f'ML-ready rows: {int(df_out["ml_ready"].sum())}')
print()
print('Arrivals stats:')
print(df_out['tourist_arrivals'].describe().round(0))
print()
print('DONE  Notebook 01 complete. Proceed to 02_eda.ipynb')